In [1]:
!wget https://norvig.com/big.txt

--2026-05-11 17:06:45--  https://norvig.com/big.txt
Resolving norvig.com (norvig.com)... 158.106.138.13
Connecting to norvig.com (norvig.com)|158.106.138.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 6488666 (6.2M) [text/plain]
Saving to: ‘big.txt’

big.txt             100%[===================>]   6.19M  6.97MB/s    in 0.9s    

2026-05-11 17:06:46 (6.97 MB/s) - ‘big.txt’ saved [6488666/6488666]



In [2]:
!ls

big.txt  sample_data


In [ ]:
import re
import string
from collections import Counter
from typing import List, Set, Tuple, Dict
import difflib


In [ ]:
class SpellChecker:
    def __init__(self, dictionary_file: str = None):
        """
        Initialize the spell checker with a dictionary.

        Args:
            dictionary_file: Path to a text file containing valid words (one per line)
        """
        self.dictionary = set()
        self.word_frequency = Counter()

        if dictionary_file:
            self.load_dictionary(dictionary_file)
            print(f'File Loaded {dictionary_file}')
        else:
            # Default small dictionary for demonstration
            #pass
            self.load_default_dictionary()

    def load_dictionary(self, file_path: str):
        """Load dictionary from a file."""
        try:
            #Opens the file, reads lines, lowercases them, strips whitespace,
            with open(file_path, 'r', encoding='utf-8') as f:
                words = [line.strip().lower() for line in f if line.strip()]
                #Adds all words into a set.
                self.dictionary = set(words)
                # Simple frequency model (all words have equal weight)
                self.word_frequency = Counter({word: 1 for word in words})
        except FileNotFoundError:
            print(f"Dictionary file {file_path} not found. Using default dictionary.")
            self.load_default_dictionary()

    def load_default_dictionary(self):
        """Load a basic dictionary for demonstration.Just hardcodes 117 English words for testing."""
        words = [
            'hello', 'world', 'python', 'programming', 'computer', 'science',
            'algorithm', 'data', 'structure', 'function', 'variable', 'loop',
            'condition', 'class', 'object', 'method', 'string', 'integer',
            'float', 'boolean', 'list', 'dictionary', 'tuple', 'set',
            'import', 'from', 'def', 'return', 'print', 'input', 'output',
            'file', 'read', 'write', 'open', 'close', 'error', 'exception',
            'try', 'except', 'finally', 'with', 'as', 'if', 'else', 'elif',
            'for', 'while', 'break', 'continue', 'pass', 'lambda', 'yield',
            'the', 'and', 'or', 'not', 'in', 'is', 'this', 'that', 'these',
            'those', 'a', 'an', 'to', 'of', 'for', 'with', 'by', 'from',
            'about', 'into', 'through', 'during', 'before', 'after', 'above',
            'below', 'up', 'down', 'out', 'off', 'over', 'under', 'again',
            'further', 'then', 'once', 'here', 'there', 'when', 'where',
            'why', 'how', 'all', 'any', 'both', 'each', 'few', 'more',
            'most', 'other', 'some', 'such', 'only', 'own', 'same', 'so',
            'than', 'too', 'very', 'can', 'will', 'just', 'should', 'now'
        ]
        self.dictionary = set(words)
        self.word_frequency = Counter({word: 1 for word in words})

    def is_valid_word(self, word: str) -> bool:
        """Check if a word exists in the dictionary."""
        return word.lower() in self.dictionary

    def extract_words(self, text: str) -> List[str]:
        """Extract words from text, removing punctuation."""
        # Remove punctuation and split into words
        words = re.findall(r'\b[a-zA-Z]+\b', text.lower())
        return words

    def edit_distance_1(self, word: str) -> Set[str]:
        """Generate all possible words with edit distance of 1."""
        letters = string.ascii_lowercase
        # splits(apple) generates: [('', 'apple'),('a', 'pple'),('ap', 'ple'),('app', 'le'),('appl', 'e'),('apple', '')]
        splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
        #For each (L, R) in splits where R is not empty:('', 'apple') → 'pple', ('a', 'pple') → 'aple', ('ap', 'ple') → 'aple', ('app', 'le') → 'appe', ('appl', 'e') → 'appl', ('apple', '') → 'apple'
        # Deletions
        deletes = [L + R[1:] for L, R in splits if R]
        #swap the positions of the first two characters in the original word word for each tuple (L, R) where len(R) > 1.
        # Transpositions
        transposes = [L + R[1] + R[0] + R[2:] for L, R in splits if len(R) > 1]

        # Replacements
        replaces = [L + c + R[1:] for L, R in splits if R for c in letters]

        # Insertions
        inserts = [L + c + R for L, R in splits for c in letters]

        return set(deletes + transposes + replaces + inserts)

    def edit_distance_2(self, word: str) -> Set[str]:
        """Generate all possible words with edit distance of 2."""
        #Take every word e1 that’s one edit away from word and call edit_distance_1(e1) again to find all words one more edit away from e1.
        return set(e2 for e1 in self.edit_distance_1(word)
                  for e2 in self.edit_distance_1(e1))

    def get_candidates(self, word: str, max_distance: int = 2) -> List[Tuple[str, float]]:
        """
        Get correction candidates for a misspelled word.

        Returns:
            List of tuples (candidate_word, confidence_score)
        """
        # If the input word is already in the dictionary there’s no correction needed it’s 100% correct.
        if self.is_valid_word(word):
            return [(word, 1.0)]

        candidates = []

        # Try edit distance 1 first, filter them to only keep those that exist in the dictionary.
        edit1_candidates = self.edit_distance_1(word)
        valid_edit1 = [w for w in edit1_candidates if self.is_valid_word(w)]

        if valid_edit1:
            # for each valid candidate Score based on frequency and similarity
            for candidate in valid_edit1:
                score = self.calculate_score(word, candidate, edit_distance=1)
                candidates.append((candidate, score))

        # If no edit distance 1 candidates and max_distance >= 2, try edit distance 2
        if not candidates and max_distance >= 2:
            edit2_candidates = self.edit_distance_2(word)
            valid_edit2 = [w for w in edit2_candidates if self.is_valid_word(w)]

            for candidate in valid_edit2:
                score = self.calculate_score(word, candidate, edit_distance=2)
                candidates.append((candidate, score))

        # Add candidates based on substring matching, Even if a word isn’t within edit distance 2, some words might still contain the misspelled part.
        substring_candidates = self.get_substring_candidates(word)
        candidates.extend(substring_candidates)

        # Remove duplicates and sort by score
        unique_candidates = {}
        for candidate, score in candidates:
            #Checks if this is the first time this candidate has been seen, compare the current score with the existing score for that candidate. If the current score is higher, it means we've found a better match.
            if candidate not in unique_candidates or score > unique_candidates[candidate]:
                unique_candidates[candidate] = score

        # Convert back to list of tuples and sort by score (descending)
        result = [(word, score) for word, score in unique_candidates.items()]
        #ex ('Apple', 95), descending order
        result.sort(key=lambda x: x[1], reverse=True)

        return result[:10]  # Return top 10 candidates

    def calculate_score(self, original: str, candidate: str, edit_distance: int) -> float:
        #ex ("applr", "apple", 1)→ 0.83
        """Calculate confidence score for a candidate."""
        # Base score inversely related to edit distance,If the edit distance is small, the base score is high.
        base_score = 1.0 / (edit_distance + 1)

        # Frequency bonus, If a candidate appears frequently in dictionary, it gets an extra boost if it’s not found, assume frequency 1
        freq_score = self.word_frequency.get(candidate, 1) / 100.0

        # Length similarity bonus, Words with similar lengths are preferred. If difference = 0 → full score (1.0) if lengths differ by 2 → lower score (1 / (2 + 1) = 0.33)
        len_diff = abs(len(original) - len(candidate))
        len_score = 1.0 / (len_diff + 1)

        # Character overlap bonus,Converts both words to sets of unique characters finds overlap.
        overlap = len(set(original) & set(candidate))
        overlap_score = overlap / max(len(set(original)), len(set(candidate)))
        # Each part contributes differently to the final score, base score is main indicator
        return base_score * 0.5 + freq_score * 0.2 + len_score * 0.2 + overlap_score * 0.1

    def get_substring_candidates(self, word: str) -> List[Tuple[str, float]]:
        """Get candidates based on substring matching."""
        candidates = []

        # Find words that contain the input as substring or vice versa
        for dict_word in self.dictionary:
            #the misspelled word is a substring of a valid word or word is substring of misspelled word
            if word in dict_word or dict_word in word:
                # Calculate similarity score SequenceMatcher computes how similar two strings are based on character alignment.
                similarity = difflib.SequenceMatcher(None, word, dict_word).ratio()
                if similarity > 0.6:  # Threshold for substring candidates at least 60% similar, avoids adding too many irrelevant candidates.
                    score = similarity * 0.3  # Lower score for substring matches
                    candidates.append((dict_word, score))

        return candidates

    def check_text(self, text: str) -> Dict[str, List[Tuple[str, float]]]:
        """
        Check entire text and return corrections for misspelled words.

        Returns:
            Dictionary mapping misspelled words to their correction candidates
        """
        words = self.extract_words(text)
        misspelled = {}

        for word in set(words):  # Use set to avoid checking duplicates
            if not self.is_valid_word(word):
                candidates = self.get_candidates(word)
                if candidates:
                    misspelled[word] = candidates
        #returns the complete mapping of all misspelled words and their possible corrections.
        return misspelled

    def correct_text(self, text: str, auto_correct: bool = False) -> str:
        """
        Correct text by replacing misspelled words.

        Args:
            text: Input text to correct
            auto_correct: If True, automatically use the best candidate

        Returns:
            Corrected text
        """
        corrections = self.check_text(text)
        #start with original text
        corrected_text = text
        # iterate over each entry in the dictionary returned by check_text()
        for misspelled_word, candidates in corrections.items():
            #If a word has no valid correction suggestions, skip it
            if candidates:
                if auto_correct:
                    # Use the best candidate, ex: candidates = [('apple', 0.92), ('apply', 0.75)] best_candidate = 'apple'
                    best_candidate = candidates[0][0]
                    #re.escape prevents regex from interpreting symbols as patterns.
                    corrected_text = re.sub(r'\b' + re.escape(misspelled_word) + r'\b',
                                          best_candidate, corrected_text, flags=re.IGNORECASE)
                else:
                    print(f"Misspelled: '{misspelled_word}'")
                    print("Suggestions:")
                    # prints each misspelled word and the top 5 correction suggestions with their confidence scores.
                    for i, (candidate, score) in enumerate(candidates[:5], 1):
                        print(f"  {i}. {candidate} (confidence: {score:.3f})")
                    print()

        return corrected_text


In [ ]:
# Example usage and demonstration
def main():
    # Initialize spell checker
    #spell_checker = SpellChecker(dictionary_file='big.txt')
    spell_checker = SpellChecker()

    print("=== Spell Checker Demo ===\n")

    # Test individual words
    test_words = ["helo", "wrold", "programing", "comuter", "algorithim"]

    print("Individual word corrections:")
    print("-" * 40)
    for word in test_words:
        candidates = spell_checker.get_candidates(word)
        print(f"'{word}' -> {candidates[:3]}")  # Show top 3 candidates

    print("\n" + "="*50 + "\n")

    # Test full text
    test_text = "Helo wrold! This is a smple text with som misspeled words for testng."

    print("Text correction:")
    print("-" * 20)
    print(f"Original: {test_text}")
    print()

    # Check for misspellings
    misspellings = spell_checker.check_text(test_text)
    print("Detected misspellings and suggestions:")
    for word, candidates in misspellings.items():
        print(f"'{word}' -> {[c[0] for c in candidates[:3]]}")

    print()

    # Auto-correct the text
    corrected = spell_checker.correct_text(test_text, auto_correct=True)
    print(f"Auto-corrected: {corrected}")

    print("\n" + "="*50 + "\n")

    # Interactive mode example
    print("Interactive correction example:")
    print("-" * 30)
    interactive_text = "I love programing in pythn!"
    print(f"Text: {interactive_text}")
    print("\nSuggestions:")
    spell_checker.correct_text(interactive_text, auto_correct=False)


In [ ]:
main()

=== Spell Checker Demo ===

Individual word corrections:
----------------------------------------
'helo' -> [('hello', 0.45199999999999996)]
'wrold' -> [('world', 0.552)]
'programing' -> [('programming', 0.45199999999999996)]
'comuter' -> [('computer', 0.4395)]
'algorithim' -> [('algorithm', 0.45199999999999996)]


Text correction:
--------------------
Original: Helo wrold! This is a smple text with som misspeled words for testng.

Detected misspellings and suggestions:
'smple' -> ['tuple']
'som' -> ['some', 'so']
'words' -> ['world']
'text' -> ['that', 'set']
'wrold' -> ['world']
'helo' -> ['hello']

Auto-corrected: hello world! This is a tuple that with some misspeled world for testng.


Interactive correction example:
------------------------------
Text: I love programing in pythn!

Suggestions:
Misspelled: 'programing'
Suggestions:
  1. programming (confidence: 0.452)

Misspelled: 'i'
Suggestions:
  1. a (confidence: 0.452)
  2. is (confidence: 0.402)
  3. in (confidence: 0.402)
  